In [1]:
!pip install streamlit pyngrok pyjwt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 23.1 MB/s eta 0:00:00
  Attempting uninstall: cachetools
    Found existing installation: cachetools 7.0.0
    Uninstalling cachetools-7.0.0:
      Successfully uninstalled cachetools-7.0.0


In [2]:
%%writefile app.py
import streamlit as st
import sqlite3
import jwt
import datetime
import re
import hashlib

# --- CONFIGURATION ---
SECRET_KEY = "my_super_secret_key_123"  # In production, keep this safe!
DB_NAME = "users.db"

# --- CUSTOM CSS ---
st.markdown("""
    <style>
    .main {
        background-color: #f0f2f6;
    }
    .stButton>button {
        width: 100%;
        background-color: #4CAF50;
        color: white;
        border-radius: 5px;
        border: none;
        padding: 10px;
    }
    .stButton>button:hover {
        background-color: #45a049;
    }
    .title-text {
        color: #2c3e50;
        text-align: center;
        font-family: 'Arial', sans-serif;
    }
    .success-msg {
        padding: 10px;
        background-color: #d4edda;
        color: #155724;
        border-radius: 5px;
        margin-bottom: 10px;
    }
    .error-msg {
        padding: 10px;
        background-color: #f8d7da;
        color: #721c24;
        border-radius: 5px;
        margin-bottom: 10px;
    }
    </style>
""", unsafe_allow_html=True)

# --- DATABASE SETUP ---
def init_db():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute('''
        CREATE TABLE IF NOT EXISTS users (
            username TEXT,
            email TEXT PRIMARY KEY,
            password TEXT,
            security_question TEXT,
            security_answer TEXT
        )
    ''')
    conn.commit()
    conn.close()

init_db()

# --- HELPER FUNCTIONS ---
def validate_email(email):
    # Matches: chars@chars.2+chars
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return re.match(pattern, email) is not None

def hash_password(password):
    return hashlib.sha256(password.encode()).hexdigest()

def create_jwt(email):
    payload = {
        'email': email,
        'exp': datetime.datetime.utcnow() + datetime.timedelta(hours=1)
    }
    token = jwt.encode(payload, SECRET_KEY, algorithm='HS256')
    return token

def verify_jwt(token):
    try:
        data = jwt.decode(token, SECRET_KEY, algorithms=['HS256'])
        return data['email']
    except jwt.ExpiredSignatureError:
        return None
    except jwt.InvalidTokenError:
        return None

# --- SESSION STATE MANAGEMENT ---
if 'page' not in st.session_state:
    st.session_state['page'] = 'Login'
if 'auth_token' not in st.session_state:
    st.session_state['auth_token'] = None
if 'current_user' not in st.session_state:
    st.session_state['current_user'] = None

def navigate_to(page):
    st.session_state['page'] = page
    st.rerun() # Rerun immediately to switch pages

# --- PAGES ---

# 1. SIGNUP PAGE
def show_signup():
    st.markdown("<h1 class='title-text'>📝 Student Signup</h1>", unsafe_allow_html=True)

    with st.form("signup_form"):
        username = st.text_input("Username")
        email = st.text_input("Email ID (e.g., student@example.com)")
        password = st.text_input("Password", type="password")
        confirm_password = st.text_input("Confirm Password", type="password")

        sec_q = st.selectbox("Security Question", [
            "What is your pet name?",
            "What is your mother’s maiden name?",
            "What is your favorite teacher?"
        ])
        sec_a = st.text_input("Security Answer")

        submit = st.form_submit_button("Sign Up")

        if submit:
            # Validations
            if not all([username, email, password, confirm_password, sec_a]):
                st.error("All fields are mandatory!")
            elif not validate_email(email):
                st.error("Invalid Email Format! (e.g., name@domain.com)")
            elif not password.isalnum():
                st.error("Password must be alphanumeric (letters + numbers only)!")
            elif password != confirm_password:
                st.error("Passwords do not match!")
            else:
                try:
                    conn = sqlite3.connect(DB_NAME)
                    c = conn.cursor()
                    hashed_pw = hash_password(password)
                    c.execute("INSERT INTO users VALUES (?, ?, ?, ?, ?)",
                              (username, email, hashed_pw, sec_q, sec_a))
                    conn.commit()
                    conn.close()
                    st.success("Signup Successful! Redirecting to Login...")
                    # Generate Token (Requirement 2) - though usually done on login,
                    # we store details securely.
                    st.session_state['page'] = 'Login'
                    st.rerun()
                except sqlite3.IntegrityError:
                    st.error("Email already exists!")

    st.markdown("---")
    st.button("Already have an account? Login", on_click=lambda: navigate_to("Login"))

# 2. LOGIN PAGE
def show_login():
    st.markdown("<h1 class='title-text'>🔐 Login</h1>", unsafe_allow_html=True)

    with st.form("login_form"):
        email = st.text_input("Email ID")
        password = st.text_input("Password", type="password")
        submit = st.form_submit_button("Login")

        if submit:
            conn = sqlite3.connect(DB_NAME)
            c = conn.cursor()
            hashed_pw = hash_password(password)
            c.execute("SELECT username FROM users WHERE email=? AND password=?", (email, hashed_pw))
            user = c.fetchone()
            conn.close()

            if user:
                token = create_jwt(email)
                st.session_state['auth_token'] = token
                st.session_state['current_user'] = user[0]
                st.success("Login Successful!")
                navigate_to("Dashboard")
            else:
                st.error("Invalid Email or Password")

    st.markdown("---")
    col1, col2 = st.columns(2)
    with col1:
        st.button("Create Account", on_click=lambda: navigate_to("Signup"))
    with col2:
        st.button("Forgot Password?", on_click=lambda: navigate_to("Forgot Password"))

# 3. DASHBOARD PAGE
def show_dashboard():
    # Verify JWT Token
    email = verify_jwt(st.session_state['auth_token'])
    if not email:
        st.error("Session expired or invalid. Please login again.")
        navigate_to("Login")
        return

    st.markdown(f"<h1 class='title-text'>👋 Welcome, {st.session_state['current_user']}!</h1>", unsafe_allow_html=True)
    st.markdown("### Dashboard Area")
    st.info(f"You are logged in securely with JWT.\n\nUser Email: {email}")

    if st.button("Logout"):
        st.session_state['auth_token'] = None
        st.session_state['current_user'] = None
        navigate_to("Login")

# 4. FORGOT PASSWORD FLOW
def show_forgot_password():
    st.markdown("<h1 class='title-text'>🔑 Reset Password</h1>", unsafe_allow_html=True)

    # State to track steps in the forgot password flow
    if 'fp_step' not in st.session_state:
        st.session_state['fp_step'] = 1
    if 'fp_email' not in st.session_state:
        st.session_state['fp_email'] = ""
    if 'fp_question' not in st.session_state:
        st.session_state['fp_question'] = ""

    # Step 1: Enter Email
    if st.session_state['fp_step'] == 1:
        email = st.text_input("Enter your Email ID")
        if st.button("Verify Email"):
            conn = sqlite3.connect(DB_NAME)
            c = conn.cursor()
            c.execute("SELECT security_question FROM users WHERE email=?", (email,))
            result = c.fetchone()
            conn.close()

            if result:
                st.session_state['fp_email'] = email
                st.session_state['fp_question'] = result[0]
                st.session_state['fp_step'] = 2
                st.rerun()
            else:
                st.error("Email not found.")
        st.button("Back to Login", on_click=lambda: navigate_to("Login"))

    # Step 2: Answer Security Question
    elif st.session_state['fp_step'] == 2:
        st.write(f"**Security Question:** {st.session_state['fp_question']}")
        answer = st.text_input("Your Answer")

        if st.button("Verify Answer"):
            conn = sqlite3.connect(DB_NAME)
            c = conn.cursor()
            c.execute("SELECT security_answer FROM users WHERE email=?", (st.session_state['fp_email'],))
            correct_answer = c.fetchone()[0]
            conn.close()

            if answer == correct_answer:
                st.session_state['fp_step'] = 3
                st.rerun()
            else:
                st.error("Incorrect Answer.")

    # Step 3: Reset Password
    elif st.session_state['fp_step'] == 3:
        new_pass = st.text_input("New Password", type="password")
        confirm_pass = st.text_input("Confirm New Password", type="password")

        if st.button("Update Password"):
            if not new_pass.isalnum():
                st.error("Password must be alphanumeric!")
            elif new_pass != confirm_pass:
                st.error("Passwords do not match!")
            else:
                conn = sqlite3.connect(DB_NAME)
                c = conn.cursor()
                hashed_pw = hash_password(new_pass)
                # Secure update
                c.execute("UPDATE users SET password=? WHERE email=?", (hashed_pw, st.session_state['fp_email']))
                conn.commit()
                conn.close()
                st.success("Password Updated Successfully!")

                # Cleanup and redirect
                del st.session_state['fp_step']
                del st.session_state['fp_email']
                del st.session_state['fp_question']
                st.session_state['page'] = 'Login'
                st.rerun()

# --- MAIN CONTROLLER ---
if st.session_state['page'] == 'Signup':
    show_signup()
elif st.session_state['page'] == 'Login':
    show_login()
elif st.session_state['page'] == 'Dashboard':
    show_dashboard()
elif st.session_state['page'] == 'Forgot Password':
    show_forgot_password()

Writing app.py


In [3]:
from pyngrok import ngrok

# 1. Authenticate ngrok
# REPLACE WITH YOUR ACTUAL TOKEN
ngrok.set_auth_token("AUTH_TOKEN")

# 2. Run Streamlit in the background
!streamlit run app.py &>/dev/null&

# 3. Create the tunnel
# Using the default Streamlit port 8501
public_url = ngrok.connect(8501).public_url

print(f"🚀 Streamlit App is live at: {public_url}")

🚀 Streamlit App is live at: https://irrevocable-darell-martyrly.ngrok-free.dev
